In [ ]:
import json
from tqdm import tqdm
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/Llama_2_13B_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_Llama2.json"

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

STOP_TOKENS = ["<|im_end|>", "<|im_start|>", "</s>"]

print("Loading Llama2 model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX, n_batch=N_BATCH, n_ubatch=N_UBATCH, n_threads=N_THREADS,
    n_gpu_layers=GPU_LAYERS, main_gpu=MAIN_GPU,
    flash_attn=True, f16_kv=True, use_mmap=True, offload_kqv=True, verbose=False,
    # If your GGUF saved a chat template, keep "chatml"; otherwise change to "llama-2"
    chat_format="chatml"
)
print("Model loaded.")

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total records: {len(data)}")

results = []
for idx, item in enumerate(tqdm(data, total=len(data), desc="Inference Progress", ncols=100)):
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": input_text}
    ]

    try:
        response = llm.create_chat_completion(
            messages=messages, temperature=TEMPERATURE, top_p=TOP_P, 
            max_tokens=MAX_TOKENS, stream=False, stop=STOP_TOKENS,
        )
        prediction = response["choices"][0]["message"]["content"]
    except Exception as e:
        print(f"\nError at sample {idx+1}: {e}")
        prediction = ""

    new_item = item.copy()
    new_item["output_Llama"] = prediction
    results.append(new_item)

    # Save checkpoint every 10 records
    if (idx + 1) % 10 == 0:
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

# Final save
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nDone! Results saved to: {OUTPUT_JSON}")